# Stage C4 — PINN Ensemble Training

Dual-fuel PINN pipeline | Sandrine Schueller Mafra | PPGEM – UFPR
Supports dissertation Sec. 3.5.2.

This is where every earlier C-stage lands: C1's constraints, C2's
composite loss and calibrated weights, C3's tuned hyperparameters and
B1's architecture, trained for real — two phases (Adam, then L-BFGS)
per member, 10 members for the ensemble mean/uncertainty split described
in Sec. 3.5.2. The model builder and every loss term are copied verbatim
from B1/C2/C3.

**Framework note:** Sec. 3.5.2's L-BFGS phase isn't something Keras ships
natively (and `tensorflow_probability` is not assumed to be installed).
This notebook bridges to `scipy.optimize.minimize(method='L-BFGS-B')`:
flatten the model's weights into one vector, hand scipy a function that
returns `(loss, gradient)` for any vector it proposes, let it optimize,
unflatten the result back into the model — a standard technique in
published PINN implementations.

**Changes from the first version of this notebook** (details in each
section):
- *Model selection waits for the physics.* Early stopping, best-weight
  tracking and the learning-rate reduction start only once C2's schedule
  has brought the physics weight to 99 % of full strength
  (epoch ≈ 1210). Before, a validation minimum reached early could
  restore weights trained with ≈ 1 % of the physics weight, i.e. an
  essentially data-only model.
- *L-BFGS runs with dropout off.* L-BFGS needs a deterministic objective;
  with dropout active every evaluation differs and its line search is
  unreliable.
- *No fallbacks.* Missing or inconsistent C2/C3 outputs stop the notebook
  instead of training an untuned configuration.
- *Validation metric = plain MSE on the normalized outputs*, the same
  quantity B1, B2 and C3 select on.
- Member 0 (Section 5) is reused as the first ensemble member instead of
  being trained twice; the ensemble std uses the sample formula (M − 1).

**Runtime expectation:** 10 members x (Adam up to 5000 epochs + L-BFGS up
to 500 iterations) is the heaviest stage. Section 5 trains and inspects
one member first, so a problem shows up before all 10 are trained.

**Input:** `data/masters_data.xlsx`, `outputs/B1_selected_architecture.json`,
`outputs/C1_collocation_points.csv`, `outputs/C1_constraint_config.json`,
`outputs/C2_loss_config.json`, `outputs/C3_best_config.json`.
**Output:** every figure and result table is saved to `outputs/html/` as
`C4_<section>[_qualifier].html` (the per-member log is
`C4_06_member_log.html`). For Phase D (read by code): 
`outputs/C4_ensemble_predictions.csv`, `outputs/C4_member_log.csv`,
`outputs/C4_training_histories.csv`, `outputs/C4_run_config.json` and the
10 saved models.

## Setup

In [ ]:
import json
import time
import numpy as np
import polars as pl
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
from scipy.optimize import minimize
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("polars    ", pl.__version__)
import plotly
print("plotly    ", plotly.__version__)
print("tensorflow", tf.__version__)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "code" else Path.cwd()
RAW_PATH = PROJECT_ROOT / "data" / "masters_data.xlsx"
OUT_DIR = PROJECT_ROOT / "outputs"
SEED = 42
UNITS = {"HC": "g/kWh", "NOx": "ppm", "CO2": "%", "PM": "FSN", "eta": "fraction"}

## Color palette and output naming (shared across the whole pipeline)

`SPLIT_COLORS` (semantic role: train / validation / test / reference
value / alert / neutral) and `VARIABLE_COLORS` (identity of each of the
4 inputs and 5 outputs) are identical in every A/B/C notebook, so the
same element always has the same color in any chart of the pipeline.

Every figure or result table generated below is also saved to
`outputs/html/`, named `C4_<section>[_qualifier].html` — the number
matches the corresponding section header, so the order in which each
output was produced can be read from the file name alone.

In [ ]:
SPLIT_COLORS = {
    "train": "#B7C9DA",
    "validation": "#2B6EFF",
    "test": "#571D99",
    "reference": "#343A40",   # value transcribed from the dissertation text
    "alert": "#E85D04",       # outlier / out of range / anomaly
    "neutral": "#B0AFA8",     # grid lines / neutral reference
}
VARIABLE_COLORS = {
    "SOI": "#073b3a", "lambda": "#0b6e4f", "sub_rate": "#08a045", "P_rail": "#6bbf59",
    "NOx": "#c7adff", "PM": "#916dd5", "eta": "#7151a9", "HC": "#573d7f", "CO2": "#46325d",
}

HTML_DIR = OUT_DIR / "html"
HTML_DIR.mkdir(parents=True, exist_ok=True)


def flagged_table_html(df, title, out_path, flag_col=None, is_flagged=lambda v: False, ref_cols=()):
    """Result table -> Plotly go.Table -> HTML.
    Columns listed in ref_cols get the 'reference' tone in the header
    (values transcribed from the dissertation text). Cells in flag_col
    get the 'alert' tone wherever is_flagged(value) is True."""
    cols = list(df.columns)
    n = df.shape[0]
    header_fill = [SPLIT_COLORS["reference"] if c in ref_cols else "#F1F3F5" for c in cols]
    header_font = ["white" if c in ref_cols else "black" for c in cols]
    cell_fill = []
    for c in cols:
        if c == flag_col:
            cell_fill.append([SPLIT_COLORS["alert"] if is_flagged(v) else "white"
                               for v in df[c].to_list()])
        else:
            cell_fill.append(["white"] * n)
    fig = go.Figure(data=[go.Table(
        header=dict(values=cols, fill_color=header_fill,
                     font=dict(color=header_font), align="left"),
        cells=dict(values=[df[c].to_list() for c in cols],
                    fill_color=cell_fill, align="left"),
    )])
    fig.update_layout(title=title, margin=dict(t=40, l=10, r=10, b=10))
    fig.write_html(str(out_path), include_plotlyjs="inline")
    return fig


def simple_table_html(df, title, out_path):
    return flagged_table_html(df, title, out_path)

# Constraint identity (not covered by the two palettes above): each constraint takes the
# color of the output it constrains, and the dash pattern tells same-output constraints apart.
CONSTRAINT_STYLE = {
    "NOx-SOI": dict(color=VARIABLE_COLORS["NOx"], dash="solid"),
    "PM-lambda": dict(color=VARIABLE_COLORS["PM"], dash="solid"),
    "HC-lambda": dict(color=VARIABLE_COLORS["HC"], dash="solid"),
    "NOx-PM": dict(color=VARIABLE_COLORS["PM"], dash="dash"),
    "eta-NOx": dict(color=VARIABLE_COLORS["eta"], dash="solid"),
    "non-negativity": dict(color=SPLIT_COLORS["reference"], dash="dot"),
}

## 1. Load data, collocation points and upstream configurations

C2's loading code, copied verbatim (same split and normalization, same
checks against C1's configuration and B1's architecture), then the
validation/test arrays and the C2/C3 outputs, with three consistency
checks: C2 was calibrated with the current C1 configuration, C3 ran
against the current C2 calibration, and C3 used B1's architecture with
`tanh` (the design decision recorded in C3's header).

In [ ]:
COLUMN_MAP = {
    "SOI [o.CA]": "SOI", "Lambda [-]": "lambda", "Sub. Rate [%]": "sub_rate",
    "Prail [bar]": "P_rail", "HC [g/kW.h]": "HC", "NOX [ppm]": "NOx",
    "CO2 [%]": "CO2", "SO_H [FSN]": "PM", "ETA [%]": "eta",
}
INPUT_COLS = ["SOI", "lambda", "sub_rate", "P_rail"]
OUTPUT_COLS = ["HC", "NOx", "CO2", "PM", "eta"]
ALL_COLS = INPUT_COLS + OUTPUT_COLS
N_IN, N_OUT = len(INPUT_COLS), len(OUTPUT_COLS)
SOI_IDX, LAMBDA_IDX, SUBRATE_IDX, PRAIL_IDX = [INPUT_COLS.index(c) for c in INPUT_COLS]
HC_IDX, NOX_IDX, CO2_IDX, PM_IDX, ETA_IDX = [OUTPUT_COLS.index(c) for c in OUTPUT_COLS]
EMISSION_IDXS = [HC_IDX, NOX_IDX, CO2_IDX, PM_IDX]

df = pl.read_excel(RAW_PATH).rename(COLUMN_MAP).select(ALL_COLS)
n = df.shape[0]
medians = {c: df[c].median() for c in INPUT_COLS}
ranges = {c: (df[c].max() - df[c].min()) for c in INPUT_COLS}
deviation = np.column_stack([np.abs(df[c].to_numpy() - medians[c]) / ranges[c] for c in INPUT_COLS])
raw_block = np.array(INPUT_COLS)[deviation.argmax(axis=1)]

def smooth_isolated_labels(labels, passes=2):
    out = list(labels)
    for _ in range(passes):
        changed = False
        for i in range(1, len(out) - 1):
            if out[i] != out[i - 1] and out[i - 1] == out[i + 1]:
                out[i] = out[i - 1]
                changed = True
        if not changed:
            break
    return np.array(out)

ofat_block = smooth_isolated_labels(raw_block)
extremity = np.zeros(n)
for b in np.unique(ofat_block):
    idx = np.where(ofat_block == b)[0]
    vals = df[b].to_numpy()[idx]
    order = np.argsort(vals)
    m = len(idx)
    pos = np.array([0.5]) if m == 1 else np.empty(m)
    if m > 1:
        ranks = np.empty(m)
        ranks[order] = np.arange(m)
        pos = ranks / (m - 1)
    extremity[idx] = np.abs(pos - 0.5) * 2
rng_np = np.random.default_rng(SEED)
jitter = rng_np.uniform(-1e-9, 1e-9, size=n)
order = np.argsort(-(extremity + jitter))
split = np.array(["train"] * n)
split[order[:6]] = "test"
split[order[6:12]] = "val"
df = df.with_columns(pl.Series("split", split))
train_df = df.filter(pl.col("split") == "train")
train_min = {c: train_df[c].min() for c in ALL_COLS}
train_max = {c: train_df[c].max() for c in ALL_COLS}
df = df.with_columns([
    ((pl.col(c) - train_min[c]) / (train_max[c] - train_min[c])).alias(f"{c}_norm")
    for c in ALL_COLS
])
X_train = df.filter(pl.col("split") == "train").select([f"{c}_norm" for c in INPUT_COLS]).to_numpy().astype(np.float32)
Y_train = df.filter(pl.col("split") == "train").select([f"{c}_norm" for c in OUTPUT_COLS]).to_numpy().astype(np.float32)

# ---- C1 outputs: collocation points + constraint configuration ----
colloc_path = OUT_DIR / "C1_collocation_points.csv"
config_path = OUT_DIR / "C1_constraint_config.json"
for p in (colloc_path, config_path):
    if not p.exists():
        raise FileNotFoundError(f"{p} not found -- run C1 (including its save cell) before C2.")
colloc_df = pl.read_csv(colloc_path)
X_colloc = colloc_df.select(INPUT_COLS).to_numpy().astype(np.float32)
rho_colloc = colloc_df["density_ratio"].to_numpy()
validity_eff_nox = colloc_df["validity_eff_nox"].to_numpy()
with open(config_path) as f:
    C1_CONFIG = json.load(f)

# the non-negativity floors must be the ones C1 computed from this same split
EMISSIONS = [OUTPUT_COLS[i] for i in EMISSION_IDXS]
floors_here = [float(-train_min[c] / (train_max[c] - train_min[c])) for c in EMISSIONS]
floors_c1 = [C1_CONFIG["nonneg_floor_norm"][c] for c in EMISSIONS]
if not np.allclose(floors_here, floors_c1):
    raise ValueError("non-negativity floors differ from C1 -- the data or split changed; rerun C1.")

# ---- B1 architecture ----
selection_path = OUT_DIR / "B1_selected_architecture.json"
if not selection_path.exists():
    raise FileNotFoundError(f"{selection_path} not found -- run B1 (including its save cell) before C2.")
with open(selection_path) as f:
    selection = json.load(f)
HIDDEN_UNITS = tuple(selection["hidden_units"])
print(f"Loaded {X_colloc.shape[0]} collocation points from C1; architecture {selection['architecture_id']} {HIDDEN_UNITS}")
print(f"C1 config: NOx-SOI direction = {C1_CONFIG['nox_soi_direction']}, validity rule = {C1_CONFIG['validity_eff_nox_rule']}")

def split_arrays(split_name, cols):
    sub = df.filter(pl.col("split") == split_name)
    return sub.select([f"{c}_norm" for c in cols]).to_numpy().astype(np.float32)

X_val, Y_val = split_arrays("val", INPUT_COLS), split_arrays("val", OUTPUT_COLS)
X_test, Y_test = split_arrays("test", INPUT_COLS), split_arrays("test", OUTPUT_COLS)
print(f"train={X_train.shape[0]}  val={X_val.shape[0]}  test={X_test.shape[0]}")

c2_path, c3_path = OUT_DIR / "C2_loss_config.json", OUT_DIR / "C3_best_config.json"
for p, stage in [(c2_path, "C2"), (c3_path, "C3")]:
    if not p.exists():
        raise FileNotFoundError(f"{p} not found -- run {stage} (including its save cell) before C4.")
with open(c2_path) as f:
    loss_config = json.load(f)
with open(c3_path) as f:
    hp = json.load(f)
if loss_config["c1_config"] != C1_CONFIG:
    raise ValueError("C2 was calibrated with a different C1 configuration -- rerun C2 (and C3).")
if hp.get("c2_calibration_mode") != loss_config["calibration_mode"]:
    raise ValueError("C3 ran against a different C2 calibration -- rerun C3.")
if list(hp["hidden_units"]) != list(HIDDEN_UNITS) or hp["activation"] != "tanh":
    raise ValueError("C3's architecture differs from B1's selection with tanh -- the design decision is to "
                     "reuse B1's architecture; rerun C3 with SEARCH_ARCHITECTURE = False.")
print(f"C3 configuration: CV mean MSE = {hp['cv_mean_mse']:.5f}, architecture from {hp['architecture_source']}")

## 2. Model and loss machinery (copied from B1/C2/C3)

The builder is C3's (B1's model + dropout), the constraint functions and
per-point physics terms are C2's, all copied verbatim. The composite loss
applies C3's category scales to C2's calibrated weights and C2's schedule
at its original time scale (Eq. 3.16, 3.20):

$$
\mathcal{L}_{\text{total}}(t) = \mathcal{L}_{\text{data}} + \sum_j \frac{w_j^{\text{final}}\, s_{c(j)}}{1 + e^{-k(t - t_0)}}\,\mathcal{L}_{\text{physics},j} + \text{wd}\cdot\mathcal{L}_{\text{reg}}
$$

with the weight decay `wd` tuned by C3. The tables confirm the model is
B1's and list the physics weight each constraint actually trains with.

In [ ]:
assert OUTPUT_COLS[-1] == "eta", "the eta head is appended last; OUTPUT_COLS must end with eta"

# eta as a physical fraction (A1 Section 7: stored as a fraction) and its train-only range (A3 Section 9)
to_fraction = (lambda v: v / 100) if df["eta"].max() > 1 else (lambda v: v)
ETA_MIN, ETA_MAX = to_fraction(train_min["eta"]), to_fraction(train_max["eta"])
ETA_MEAN_TRAIN = float(np.mean(to_fraction(df.filter(pl.col("split") == "train")["eta"].to_numpy())))
ETA_BIAS_INIT = float(np.log(ETA_MEAN_TRAIN / (1 - ETA_MEAN_TRAIN)))    # logit(mean train eta)


def build_model(hidden_units, seed, input_dim=N_IN, output_dim=N_OUT):
    tf.random.set_seed(seed)
    inputs = keras.Input(shape=(input_dim,))
    x = inputs
    for units in hidden_units:
        x = layers.Dense(units, activation="tanh")(x)
    emissions = layers.Dense(output_dim - 1, activation="linear", name="emissions")(x)   # HC, NOx, CO2, PM
    eta_frac = layers.Dense(1, activation="sigmoid", name="eta_fraction",
                            bias_initializer=keras.initializers.Constant(ETA_BIAS_INIT))(x)
    eta_norm = layers.Rescaling(scale=1.0 / (ETA_MAX - ETA_MIN),
                                offset=-ETA_MIN / (ETA_MAX - ETA_MIN), name="eta_rescaled")(eta_frac)
    outputs = layers.Concatenate(name="outputs")([emissions, eta_norm])
    model = keras.Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer="adam", loss="mse")
    return model


def predict_np(model, X):
    """Forward pass as a NumPy array, without model.predict().

    model.predict() builds a new tf.function for every freshly built model;
    in a loop of 240 fits that triggers TensorFlow's "tf.function retracing"
    warning and is slower than a direct call on arrays this small. A direct
    call with training=False gives the same predictions (no dropout or batch
    normalization in these models)."""
    return np.asarray(model(np.asarray(X, dtype="float32"), training=False))


def build_model_hp(hidden_units, activation, dropout_rate, seed, input_dim=N_IN, output_dim=N_OUT):
    tf.random.set_seed(seed)
    inputs = keras.Input(shape=(input_dim,))
    x = inputs
    for units in hidden_units:
        x = layers.Dense(units, activation=activation)(x)
        if dropout_rate > 0:
            x = layers.Dropout(dropout_rate)(x)
    emissions = layers.Dense(output_dim - 1, activation="linear", name="emissions")(x)
    eta_frac = layers.Dense(1, activation="sigmoid", name="eta_fraction",
                            bias_initializer=keras.initializers.Constant(ETA_BIAS_INIT))(x)
    eta_norm = layers.Rescaling(scale=1.0 / (ETA_MAX - ETA_MIN),
                                offset=-ETA_MIN / (ETA_MAX - ETA_MIN), name="eta_rescaled")(eta_frac)
    outputs = layers.Concatenate(name="outputs")([emissions, eta_norm])
    return keras.Model(inputs=inputs, outputs=outputs)



def build_member(seed):
    return build_model_hp(hp["hidden_units"], hp["activation"], hp["dropout_rate"], seed=seed)


_probe = build_member(seed=0)
model_check = pl.DataFrame({
    "check": ["architecture (B1)", "hidden units", "activation", "dropout rate (C3)",
              "trainable parameters", "parameters recorded by B1", "matches B1"],
    "value": [str(selection["architecture_id"]), str(list(HIDDEN_UNITS)), hp["activation"],
              f"{hp['dropout_rate']:.4f}", str(_probe.count_params()), str(selection["n_params"]),
              str(_probe.count_params() == selection["n_params"])],
})
flagged_table_html(model_check, "C4 -- PINN model vs. B1 selection", HTML_DIR / "C4_02_model_check.html",
                    flag_col="value", is_flagged=lambda v: v == "False")
assert _probe.count_params() == selection["n_params"]
model_check

In [ ]:
# ---- C1's constraint settings (from C1_constraint_config.json) ----
NOX_SOI_DIRECTION = C1_CONFIG["nox_soi_direction"]
PM_LAMBDA_DIRECTION = C1_CONFIG["pm_lambda_direction"]
NONNEG_FLOOR_NORM = floors_c1

# ---- C1's constraint functions, copied verbatim ----
def monotonic_constraint(predict_fn, x, out_idx, in_idx, direction):
    x_t = tf.convert_to_tensor(x, dtype=tf.float32)
    with tf.GradientTape() as tape:
        tape.watch(x_t)
        y = predict_fn(x_t)
        target = y[:, out_idx]
    grad = tape.gradient(target, x_t)
    d = grad[:, in_idx]
    violation = d if direction == "decreasing" else -d    # the derivative sign that is NOT allowed
    return tf.reduce_mean(tf.square(tf.maximum(0.0, violation)))


def nox_soi_constraint(predict_fn, x, direction=None):
    return monotonic_constraint(predict_fn, x, NOX_IDX, SOI_IDX, direction or NOX_SOI_DIRECTION)


def pm_lambda_constraint(predict_fn, x):
    return monotonic_constraint(predict_fn, x, PM_IDX, LAMBDA_IDX, PM_LAMBDA_DIRECTION)


def convexity_constraint(predict_fn, x, out_idx, in_idx):
    x_t = tf.convert_to_tensor(x, dtype=tf.float32)
    with tf.GradientTape() as tape2:
        tape2.watch(x_t)
        with tf.GradientTape() as tape1:
            tape1.watch(x_t)
            y = predict_fn(x_t)
            target = y[:, out_idx]
        grad1 = tape1.gradient(target, x_t)
        d_first = grad1[:, in_idx]
    grad2 = tape2.gradient(d_first, x_t)
    d_second = grad2[:, in_idx]
    return tf.reduce_mean(tf.square(tf.maximum(0.0, -d_second)))


def hc_lambda_constraint(predict_fn, x):
    return convexity_constraint(predict_fn, x, HC_IDX, LAMBDA_IDX)


def tradeoff_constraint(predict_fn, x, out_idx_a, out_idx_b, in_idx):
    x_t = tf.convert_to_tensor(x, dtype=tf.float32)
    with tf.GradientTape(persistent=True) as tape:
        tape.watch(x_t)
        y = predict_fn(x_t)
        a = y[:, out_idx_a]
        b = y[:, out_idx_b]
    grad_a = tape.gradient(a, x_t)[:, in_idx]
    grad_b = tape.gradient(b, x_t)[:, in_idx]
    del tape
    return tf.reduce_mean(tf.square(tf.maximum(0.0, grad_a * grad_b)))


def nox_pm_tradeoff_constraint(predict_fn, x):
    return tradeoff_constraint(predict_fn, x, NOX_IDX, PM_IDX, SOI_IDX)


def eff_nox_tradeoff_constraint(predict_fn, x):
    return tradeoff_constraint(predict_fn, x, ETA_IDX, NOX_IDX, SOI_IDX)


def nonneg_constraint(predict_fn, x, out_idxs=EMISSION_IDXS, floors=NONNEG_FLOOR_NORM):
    x_t = tf.convert_to_tensor(x, dtype=tf.float32)
    y = predict_fn(x_t)
    emissions = tf.gather(y, out_idxs, axis=1)
    floor_t = tf.constant(floors, dtype=tf.float32)          # normalized value of physical zero
    # Eq. 3.14: sum over the four emissions, mean over points
    return tf.reduce_mean(tf.reduce_sum(tf.square(tf.maximum(0.0, floor_t - emissions)), axis=1))

In [ ]:
def g_monotonic(predict_fn, x, out_idx, in_idx, direction):
    x_t = tf.convert_to_tensor(x, dtype=tf.float32)
    with tf.GradientTape() as tape:
        tape.watch(x_t)
        target = predict_fn(x_t)[:, out_idx]
    d = tape.gradient(target, x_t)[:, in_idx]
    return d if direction == "decreasing" else -d

def g_convexity(predict_fn, x, out_idx, in_idx):
    x_t = tf.convert_to_tensor(x, dtype=tf.float32)
    with tf.GradientTape() as tape2:
        tape2.watch(x_t)
        with tf.GradientTape() as tape1:
            tape1.watch(x_t)
            target = predict_fn(x_t)[:, out_idx]
        d_first = tape1.gradient(target, x_t)[:, in_idx]
    d_second = tape2.gradient(d_first, x_t)[:, in_idx]
    return -d_second

def g_tradeoff(predict_fn, x, out_idx_a, out_idx_b, in_idx):
    x_t = tf.convert_to_tensor(x, dtype=tf.float32)
    with tf.GradientTape(persistent=True) as tape:
        tape.watch(x_t)
        y = predict_fn(x_t)
        a, b = y[:, out_idx_a], y[:, out_idx_b]
    grad_a = tape.gradient(a, x_t)[:, in_idx]
    grad_b = tape.gradient(b, x_t)[:, in_idx]
    del tape
    return grad_a * grad_b

def g_nonneg(predict_fn, x, out_idxs=EMISSION_IDXS, floors=NONNEG_FLOOR_NORM):
    x_t = tf.convert_to_tensor(x, dtype=tf.float32)
    emissions = tf.gather(predict_fn(x_t), out_idxs, axis=1)
    return tf.constant(floors, dtype=tf.float32) - emissions        # [N, 4]

G_FUNCTIONS = {
    "NOx-SOI": lambda f, x: g_monotonic(f, x, NOX_IDX, SOI_IDX, NOX_SOI_DIRECTION),
    "PM-lambda": lambda f, x: g_monotonic(f, x, PM_IDX, LAMBDA_IDX, PM_LAMBDA_DIRECTION),
    "HC-lambda": lambda f, x: g_convexity(f, x, HC_IDX, LAMBDA_IDX),
    "NOx-PM": lambda f, x: g_tradeoff(f, x, NOX_IDX, PM_IDX, SOI_IDX),
    "eta-NOx": lambda f, x: g_tradeoff(f, x, ETA_IDX, NOX_IDX, SOI_IDX),
    "non-negativity": lambda f, x: g_nonneg(f, x),
}
CONSTRAINT_NAMES = list(G_FUNCTIONS)
GRADIENT_CONSTRAINTS = CONSTRAINT_NAMES[:5]          # these carry lambda_j(x); non-negativity does not

def per_point_penalty(g):
    p = tf.square(tf.maximum(0.0, g))
    return tf.reduce_sum(p, axis=1) if len(g.shape) > 1 else p

def per_point_magnitude(g):
    m = tf.square(g)
    return tf.reduce_sum(m, axis=1) if len(g.shape) > 1 else m

VALIDITY = {name: np.ones(len(X_colloc)) for name in GRADIENT_CONSTRAINTS}
VALIDITY["eta-NOx"] = validity_eff_nox
LAMBDA_X = {name: rho_colloc * VALIDITY[name] for name in GRADIENT_CONSTRAINTS}   # lambda_j^0 = 1 here

def physics_loss(name, predict_fn, x=X_colloc):
    p = per_point_penalty(G_FUNCTIONS[name](predict_fn, x))
    if name in LAMBDA_X:
        return tf.reduce_mean(tf.constant(LAMBDA_X[name], dtype=tf.float32) * p)
    return tf.reduce_mean(p)

In [ ]:
sigma2 = Y_train.var(axis=0, ddof=1)
sigma2 = np.maximum(sigma2, 1e-8)  # guard against a near-constant output

def data_loss(predict_fn, X, Y, sigma2=sigma2):
    X_t = tf.convert_to_tensor(X, dtype=tf.float32)
    Y_t = tf.convert_to_tensor(Y, dtype=tf.float32)
    pred = predict_fn(X_t)
    sq_err = tf.square(pred - Y_t) / tf.constant(sigma2, dtype=tf.float32)
    return tf.reduce_mean(tf.reduce_sum(sq_err, axis=1))


def regularization_loss(model):
    P = model.count_params()
    sq_sum = tf.add_n([tf.reduce_sum(tf.square(w)) for w in model.trainable_weights if len(w.shape) > 1])
    return sq_sum / P



CATEGORY_OF = {"NOx-SOI": "monotonic", "PM-lambda": "monotonic", "HC-lambda": "shape",
               "NOx-PM": "tradeoff", "eta-NOx": "tradeoff", "non-negativity": "nonneg"}
CATEGORY_KEY = {"monotonic": "scale_monotonic", "shape": "scale_shape",
                "tradeoff": "scale_tradeoff", "nonneg": "scale_nonneg"}
scaled_w = {name: loss_config["w_final"][name] * hp[CATEGORY_KEY[CATEGORY_OF[name]]] for name in CONSTRAINT_NAMES}

K_SCHEDULE, T0_SCHEDULE = loss_config["k_schedule"], loss_config["t0_schedule"]
T99_EPOCH = T0_SCHEDULE + np.log(99) / K_SCHEDULE      # physics weight reaches 99 % of full strength


def scheduled_weight(t, w_j_final, k=K_SCHEDULE, t0=T0_SCHEDULE):
    return w_j_final / (1 + np.exp(-k * (t - t0)))


def full_loss(model, X, Y, epoch, training=True, return_components=False):
    predict_fn = lambda x: model(x, training=training)
    l_data = data_loss(predict_fn, X, Y)
    components, total = {"data": l_data}, l_data
    for name in CONSTRAINT_NAMES:
        if scaled_w[name] == 0:                 # zero weight (B2 baseline): skip, don't compute
            components[name] = 0.0
            continue
        term = float(scheduled_weight(epoch, scaled_w[name])) * physics_loss(name, predict_fn)
        components[name] = term
        total = total + term
    l_reg = hp["weight_decay"] * regularization_loss(model)
    components["reg"] = l_reg
    total = total + l_reg
    return (total, components) if return_components else total


physics_weights = pl.DataFrame([{
    "constraint": name, "category": CATEGORY_OF[name],
    "w_final (C2, " + loss_config["calibration_mode"] + ")": float(f"{loss_config['w_final'][name]:.5g}"),
    "scale (C3)": float(f"{hp[CATEGORY_KEY[CATEGORY_OF[name]]]:.5g}"),
    "w used at full strength": float(f"{scaled_w[name]:.5g}"),
} for name in CONSTRAINT_NAMES])
simple_table_html(physics_weights, f"C4 -- Physics weights (schedule k = {K_SCHEDULE}, t0 = {T0_SCHEDULE}; "
                                   f"99 % from epoch {T99_EPOCH:.0f})",
                   HTML_DIR / "C4_02_physics_weights.html")
physics_weights

## 3. Phase 1 — Adam (Sec. 3.5.2)

Up to 5000 epochs, early stopping after 200 epochs without validation
improvement, learning rate × 0.5 after 50 stagnant epochs — the full
budget (C3 used a compressed version for its many search trials).

**When model selection starts.** The physics weight follows C2's
schedule: ≈ 0.05 % at epoch 0, half at epoch 750, 99 % at epoch
$t_{99} \approx 1210$. Early stopping, best-weight tracking and the
learning-rate reduction all start at $t_{99}$:

- before it, the objective itself is still changing (the physics terms
  grow every epoch), so "no improvement for 200 epochs" doesn't mean
  convergence, and halving the learning rate would slow the network down
  right when the physics arrives;
- selecting the best validation epoch before $t_{99}$ would keep a model
  trained with a small fraction of the physics weight — the PINN would
  effectively be a data-only model.

So every member trains at least $t_{99}$ + 200 epochs. The validation
metric is plain MSE on the normalized outputs (same as B1, B2 and C3).
Every epoch logs the schedule factor, the learning rate, the training
loss by component and the validation MSE.

In [ ]:
MAX_EPOCHS = 5000
PATIENCE = 200
LR_PATIENCE = 50
LR_FACTOR = 0.5
LBFGS_MAXITER = 500
N_ENSEMBLE = 10
SELECTION_START = int(np.ceil(T99_EPOCH))
assert MAX_EPOCHS > SELECTION_START + PATIENCE, "epoch cap too low for the schedule + patience"
COMPONENTS = ["data"] + CONSTRAINT_NAMES + ["reg"]


def val_mse(model, X=X_val, Y=Y_val):
    return float(np.mean((predict_np(model, X) - Y) ** 2))


def train_adam_phase(model, seed, max_epochs=MAX_EPOCHS, patience=PATIENCE, lr_patience=LR_PATIENCE,
                     verbose_every=250):
    optimizer = keras.optimizers.Adam(learning_rate=hp["learning_rate"],
                                      beta_1=hp["beta_1"], beta_2=hp["beta_2"])
    history = {k: [] for k in ["epoch", "schedule_factor", "lr", "train_total", "val_mse"]
               + [f"train_{c}" for c in COMPONENTS]}
    best_val, best_weights, best_epoch, no_improve, plateau = np.inf, None, None, 0, 0
    rng = np.random.default_rng(seed)
    epoch = 0
    for epoch in range(max_epochs):
        idx = rng.permutation(len(X_train))
        for start in range(0, len(idx), hp["batch_size"]):
            batch = idx[start:start + hp["batch_size"]]
            with tf.GradientTape() as tape:
                loss = full_loss(model, X_train[batch], Y_train[batch], epoch, training=True)
            grads = tape.gradient(loss, model.trainable_weights)
            optimizer.apply_gradients(zip(grads, model.trainable_weights))

        total, comps = full_loss(model, X_train, Y_train, epoch, training=False, return_components=True)
        v = val_mse(model)
        factor = float(scheduled_weight(epoch, 1.0))
        history["epoch"].append(epoch)
        history["schedule_factor"].append(factor)
        history["lr"].append(float(optimizer.learning_rate))
        history["train_total"].append(float(total))
        history["val_mse"].append(v)
        for c in COMPONENTS:
            history[f"train_{c}"].append(float(comps[c]))
        if epoch % verbose_every == 0:
            print(f"    epoch {epoch:5d}  physics x{factor:.3f}  train={float(total):.5f}  "
                  f"val_mse={v:.5f}  lr={float(optimizer.learning_rate):.2e}")

        if epoch < SELECTION_START:
            continue                                  # physics not at full strength: no selection yet
        if v < best_val - 1e-7:
            best_val, best_epoch, no_improve, plateau = v, epoch, 0, 0
            best_weights = [w.numpy().copy() for w in model.trainable_weights]
        else:
            no_improve += 1
            plateau += 1
        if plateau >= lr_patience:
            optimizer.learning_rate.assign(float(optimizer.learning_rate) * LR_FACTOR)
            plateau = 0
        if no_improve >= patience:
            print(f"    early stop at epoch {epoch} (no val improvement for {patience} epochs)")
            break

    if best_weights is not None:
        for w, bw in zip(model.trainable_weights, best_weights):
            w.assign(bw)
    epochs_run = epoch + 1
    info = {"epochs_run": epochs_run, "best_epoch": best_epoch, "early_stopped": epochs_run < max_epochs,
            "best_val_mse_adam": best_val}
    return history, info

## 4. Phase 2 — L-BFGS (Sec. 3.5.2)

The scipy bridge: flatten the model's weights into one vector, minimize
`full_loss` as a function of that vector (returning both loss and
gradient — `jac=True` tells scipy the objective already provides the
gradient), unflatten the result back onto the model.

**Dropout off during L-BFGS** (`training=False`): L-BFGS builds a
curvature estimate from successive loss/gradient pairs and uses a line
search, both of which assume the same weights always give the same
loss. With dropout active every evaluation would sample a new mask.
The loss is evaluated at the final Adam epoch, where the physics weight
is at full strength. The validation MSE before and after L-BFGS is
logged for every member, since this phase minimizes the training loss
without looking at validation data.

In [ ]:
def flatten_weights(model):
    return np.concatenate([w.numpy().ravel() for w in model.trainable_weights]).astype(np.float64)


def unflatten_and_assign(model, flat_vec):
    idx = 0
    for w in model.trainable_weights:
        size = int(np.prod(w.shape))
        w.assign(flat_vec[idx:idx + size].reshape(w.shape).astype(np.float32))
        idx += size


def train_lbfgs_phase(model, final_epoch, maxiter=LBFGS_MAXITER):
    def objective(flat_vec):
        unflatten_and_assign(model, flat_vec)
        with tf.GradientTape() as tape:
            loss = full_loss(model, X_train, Y_train, final_epoch, training=False)
        grads = tape.gradient(loss, model.trainable_weights)
        flat_grads = np.concatenate([g.numpy().ravel() for g in grads]).astype(np.float64)
        return float(loss), flat_grads

    x0 = flatten_weights(model)
    result = minimize(objective, x0, jac=True, method="L-BFGS-B",
                      options={"maxiter": maxiter, "maxfun": maxiter * 2})
    unflatten_and_assign(model, result.x)
    return result


def train_member(member, seed, verbose_every=250):
    t_start = time.time()
    model = build_member(seed=seed)
    hist, info = train_adam_phase(model, seed=seed, verbose_every=verbose_every)
    final_epoch = hist["epoch"][-1]
    loss_adam = float(full_loss(model, X_train, Y_train, final_epoch, training=False))
    val_adam = val_mse(model)
    result = train_lbfgs_phase(model, final_epoch=final_epoch)
    loss_lbfgs = float(full_loss(model, X_train, Y_train, final_epoch, training=False))
    val_lbfgs = val_mse(model)
    eval_fn = lambda x: model(x, training=False)
    issues = []
    if not info["early_stopped"]:
        issues.append("hit epoch cap")
    if not result.success:
        issues.append("L-BFGS not converged")
    if loss_lbfgs > loss_adam:
        issues.append("L-BFGS raised train loss")
    if val_lbfgs > val_adam:
        issues.append("L-BFGS raised val MSE")
    record = {"member": member, "seed": seed, **info,
              "train_loss_after_adam": loss_adam, "train_loss_after_lbfgs": loss_lbfgs,
              "val_mse_after_adam": val_adam, "val_mse_after_lbfgs": val_lbfgs,
              "lbfgs_converged": bool(result.success), "lbfgs_iterations": int(result.nit),
              **{f"penalty_{n}": float(physics_loss(n, eval_fn)) for n in CONSTRAINT_NAMES},
              "duration_s": round(time.time() - t_start, 1),
              "check": "OK" if not issues else "; ".join(issues)}
    return model, hist, record

## 5. One ensemble member, end to end — sanity check before running all 10

Trains member 0 (seed 0) through both phases. The table shows its KPIs;
the figure shows every loss component and the validation MSE over the
epochs, with the start of model selection ($t_{99}$) and the selected
epoch marked — the real-training answer to the feedback question of
whether the physics terms vanish or dominate. Member 0 is then reused as
the first member of the ensemble.

In [ ]:
member0, hist0, rec0 = train_member(0, seed=0)
member_view = lambda recs: pl.DataFrame([{k: (float(f"{v:.6g}") if isinstance(v, float) else v)
                                          for k, v in r.items()} for r in recs])
member0_summary = member_view([rec0])
flagged_table_html(member0_summary, "C4 -- Member 0: training KPIs", HTML_DIR / "C4_05_member0_summary.html",
                    flag_col="check", is_flagged=lambda v: v != "OK")
member0_summary

In [ ]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=["training loss by component (member 0)", "validation MSE (member 0)"])
fig.add_trace(go.Scatter(x=hist0["epoch"], y=hist0["train_data"], mode="lines", name="data",
                         line=dict(color=SPLIT_COLORS["train"], width=3)), row=1, col=1)
for name in CONSTRAINT_NAMES:
    st = CONSTRAINT_STYLE[name]
    fig.add_trace(go.Scatter(x=hist0["epoch"], y=np.maximum(hist0[f"train_{name}"], 1e-12), mode="lines",
                             name=name, line=dict(color=st["color"], dash=st["dash"], width=1.5)), row=1, col=1)
fig.add_trace(go.Scatter(x=hist0["epoch"], y=np.maximum(hist0["train_reg"], 1e-12), mode="lines", name="reg",
                         line=dict(color=SPLIT_COLORS["neutral"], dash="dot", width=1.5)), row=1, col=1)
fig.add_trace(go.Scatter(x=hist0["epoch"], y=hist0["val_mse"], mode="lines", name="validation MSE",
                         line=dict(color=SPLIT_COLORS["validation"], width=2)), row=2, col=1)
for r in (1, 2):
    fig.add_vline(x=SELECTION_START, line_dash="dot", line_color=SPLIT_COLORS["reference"], row=r, col=1)
    if rec0["best_epoch"] is not None:
        fig.add_vline(x=rec0["best_epoch"], line_dash="dash", line_color=SPLIT_COLORS["validation"], row=r, col=1)
fig.update_yaxes(type="log", title_text="loss (floored at 1e-12)", row=1, col=1)
fig.update_yaxes(type="log", title_text="val MSE", row=2, col=1)
fig.update_xaxes(title_text="epoch", row=2, col=1)
fig.update_layout(title=f"Member 0: loss components; dotted = selection starts (epoch {SELECTION_START}), "
                        f"dashed = selected epoch ({rec0['best_epoch']})", width=950, height=720)
fig.show()
fig.write_html(str(HTML_DIR / "C4_05_member0_training.html"), include_plotlyjs="inline")

## 6. Train the remaining members

Members 1 to 9 (seeds 1–9), each through both phases, reusing member 0
from Section 5. The member log records, for every member: epochs run,
selected epoch, whether early stopping fired, training loss and
validation MSE after each phase, L-BFGS convergence and iterations, the
physics penalty per constraint at the final weights (unscaled — how well
each constraint is satisfied) and the duration. A member with an issue
(epoch cap reached, L-BFGS not converged or raising the training loss or
the validation MSE) is flagged in the `check` column.

In [ ]:
ensemble_models, ensemble_histories, member_records = [member0], [hist0], [rec0]
for m in range(1, N_ENSEMBLE):
    print(f"\n=== member {m + 1}/{N_ENSEMBLE} (seed={m}) ===")
    model, hist, rec = train_member(m, seed=m, verbose_every=1000)
    ensemble_models.append(model)
    ensemble_histories.append(hist)
    member_records.append(rec)

member_log = member_view(member_records)
OUT_DIR.mkdir(parents=True, exist_ok=True)
member_log.write_csv(OUT_DIR / "C4_member_log.csv")
flagged_table_html(member_log, f"C4 -- Member log ({N_ENSEMBLE} members)", HTML_DIR / "C4_06_member_log.html",
                    flag_col="check", is_flagged=lambda v: v != "OK")
print(f"\nAll {N_ENSEMBLE} members trained.")
member_log

## 7. Training curves, all members

Top: training loss (total) per member; bottom: validation MSE per member.
Members are not distinguished by color (they are replicates, not
categories); the markers show each member's value after L-BFGS.

In [ ]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=["training loss (total)", "validation MSE"])
for m, (hist, rec) in enumerate(zip(ensemble_histories, member_records)):
    fig.add_trace(go.Scatter(x=hist["epoch"], y=hist["train_total"], mode="lines", opacity=0.6,
                             line=dict(color=SPLIT_COLORS["train"], width=1.3), name="members",
                             showlegend=(m == 0), legendgroup="train"), row=1, col=1)
    fig.add_trace(go.Scatter(x=hist["epoch"], y=hist["val_mse"], mode="lines", opacity=0.6,
                             line=dict(color=SPLIT_COLORS["validation"], width=1.3), showlegend=False), row=2, col=1)
    fig.add_trace(go.Scatter(x=[hist["epoch"][-1]], y=[rec["train_loss_after_lbfgs"]], mode="markers",
                             marker=dict(color=SPLIT_COLORS["reference"], size=9, symbol="star"),
                             name="after L-BFGS", showlegend=(m == 0), legendgroup="lbfgs"), row=1, col=1)
    fig.add_trace(go.Scatter(x=[hist["epoch"][-1]], y=[rec["val_mse_after_lbfgs"]], mode="markers",
                             marker=dict(color=SPLIT_COLORS["reference"], size=9, symbol="star"),
                             showlegend=False), row=2, col=1)
for r in (1, 2):
    fig.add_vline(x=SELECTION_START, line_dash="dot", line_color=SPLIT_COLORS["reference"], row=r, col=1)
fig.update_yaxes(type="log", row=1, col=1)
fig.update_yaxes(type="log", row=2, col=1)
fig.update_xaxes(title_text="epoch", row=2, col=1)
fig.update_layout(title=f"All {N_ENSEMBLE} members (dotted line = model selection starts, epoch {SELECTION_START})",
                  width=950, height=700)
fig.show()
fig.write_html(str(HTML_DIR / "C4_07_training_curves.html"), include_plotlyjs="inline")

## 8. Ensemble predictions — mean and uncertainty

$$
\hat y(x) = \frac{1}{M}\sum_{m=1}^{M} \hat y_m(x), \qquad
\sigma_{\text{ens}}(x) = \sqrt{\frac{1}{M-1}\sum_{m=1}^{M}\left(\hat y_m(x) - \hat y(x)\right)^2},
\qquad M = 10
$$

(sample standard deviation; with $M = 10$ it is ≈ 5 % larger than the
population formula). Metrics on the ensemble mean, in physical units, use
the same definitions as B2 (R², RMSE, MAE), so the two tables can be
compared directly — the full Eq. 3.21–3.27 treatment is Phase D's job.
Negative R² is flagged. Validation/test have 6 points each, placed at
the domain's boundaries by A3: these numbers measure extrapolation.

In [ ]:
def ensemble_predict(X):
    preds = np.stack([predict_np(m, X) for m in ensemble_models], axis=0)  # [M, N, 5]
    return preds.mean(axis=0), preds.std(axis=0, ddof=1), preds


def to_physical(norm_values, out):
    return norm_values * (train_max[out] - train_min[out]) + train_min[out]


def to_physical_spread(norm_std, out):
    return norm_std * (train_max[out] - train_min[out])


ens = {s: ensemble_predict(X) for s, X in [("train", X_train), ("val", X_val), ("test", X_test)]}
truths = {"train": Y_train, "val": Y_val, "test": Y_test}
SPLIT_NAME_OF = {"train": "train", "val": "validation", "test": "test"}
SPLIT_COLOR_OF = {"train": SPLIT_COLORS["train"], "val": SPLIT_COLORS["validation"], "test": SPLIT_COLORS["test"]}

rows = []
for s in ["train", "val", "test"]:
    mean_s, std_s, _ = ens[s]
    Y = truths[s]
    for i, out in enumerate(OUTPUT_COLS):
        ss_res = np.sum((Y[:, i] - mean_s[:, i]) ** 2)
        ss_tot = np.sum((Y[:, i] - Y[:, i].mean()) ** 2)
        err = to_physical(mean_s[:, i], out) - to_physical(Y[:, i], out)
        rows.append({"split": SPLIT_NAME_OF[s], "output": out, "n": int(Y.shape[0]),
                     "r2": round(float(1 - ss_res / ss_tot), 4) if ss_tot > 0 else float("nan"),
                     "rmse": round(float(np.sqrt(np.mean(err ** 2))), 5),
                     "mae": round(float(np.mean(np.abs(err))), 5),
                     "mean_ensemble_std": round(float(np.mean(to_physical_spread(std_s[:, i], out))), 5),
                     "unit": UNITS[out]})
metrics = pl.DataFrame(rows)
flagged_table_html(metrics, "C4 -- Ensemble mean: R2, RMSE, MAE and mean ensemble std (physical units)",
                    HTML_DIR / "C4_08_metrics_table.html", flag_col="r2", is_flagged=lambda v: v < 0)
metrics

In [ ]:
fig = make_subplots(rows=2, cols=3, subplot_titles=[f"{o} [{UNITS[o]}]" for o in OUTPUT_COLS])
for i, out in enumerate(OUTPUT_COLS):
    r, c = divmod(i, 3)
    lo, hi = np.inf, -np.inf
    for s in ["train", "val", "test"]:
        mean_s, std_s, _ = ens[s]
        obs, prd = to_physical(truths[s][:, i], out), to_physical(mean_s[:, i], out)
        lo, hi = min(lo, obs.min(), prd.min()), max(hi, obs.max(), prd.max())
        fig.add_trace(go.Scatter(
            x=obs, y=prd, mode="markers", name=SPLIT_NAME_OF[s], showlegend=(i == 0),
            error_y=dict(type="data", array=to_physical_spread(std_s[:, i], out), visible=True,
                         color=SPLIT_COLOR_OF[s], thickness=1),
            marker=dict(color=SPLIT_COLOR_OF[s], size=8, line=dict(color=SPLIT_COLORS["reference"], width=0.5))),
            row=r + 1, col=c + 1)
    fig.add_trace(go.Scatter(x=[lo, hi], y=[lo, hi], mode="lines", showlegend=False,
                             line=dict(color=SPLIT_COLORS["neutral"], dash="dot")), row=r + 1, col=c + 1)
    fig.update_xaxes(title_text="observed", row=r + 1, col=c + 1)
    fig.update_yaxes(title_text="ensemble mean", row=r + 1, col=c + 1)
fig.update_layout(height=720, width=1050,
                  title_text="C4 PINN ensemble: predicted vs. observed (physical units, error bars = 1 ensemble std)")
fig.show()
fig.write_html(str(HTML_DIR / "C4_08_pred_vs_obs.html"), include_plotlyjs="inline")

## 9. Diversity check — did the members actually find different solutions?

If they collapsed to (near-)identical weights, the ensemble std above
is meaningless as an uncertainty estimate — it would just be numerical
noise, not genuine disagreement between local minima. Left: mean
ensemble std per output on the test points (normalized units); right:
the spread of the 10 members' predictions at each test point, one panel
per output.

In [ ]:
_, std_test, all_test = ens["test"]
spread_per_output = std_test.mean(axis=0)
fig = make_subplots(rows=2, cols=3, subplot_titles=["mean ensemble std by output (test, normalized)"]
                    + [f"member spread per test point: {o}" for o in OUTPUT_COLS])
fig.add_trace(go.Bar(x=OUTPUT_COLS, y=spread_per_output, marker_color=[VARIABLE_COLORS[o] for o in OUTPUT_COLS],
                     showlegend=False), row=1, col=1)
for k, out in enumerate(OUTPUT_COLS, start=1):
    r, c = divmod(k, 3)
    for j in range(all_test.shape[1]):
        fig.add_trace(go.Box(y=all_test[:, j, k - 1], name=f"pt {j}", marker_color=SPLIT_COLORS["test"],
                             showlegend=False), row=r + 1, col=c + 1)
fig.update_layout(height=700, width=1050, title_text="Ensemble diversity (test points)")
fig.show()
fig.write_html(str(HTML_DIR / "C4_09_diversity.html"), include_plotlyjs="inline")

spread_rows = []
for s in ["train", "val", "test"]:
    _, std_s, _ = ens[s]
    for i, out in enumerate(OUTPUT_COLS):
        v = float(std_s[:, i].mean())
        spread_rows.append({"split": SPLIT_NAME_OF[s], "output": out, "mean_std_norm": float(f"{v:.5g}"),
                            "members_collapsed (< 1e-4)": v < 1e-4})
ensemble_spread = pl.DataFrame(spread_rows)
flagged_table_html(ensemble_spread, "C4 -- Ensemble spread per split and output (normalized units)",
                    HTML_DIR / "C4_09_ensemble_spread.html",
                    flag_col="members_collapsed (< 1e-4)", is_flagged=lambda v: v)
near_zero_std = int((spread_per_output < 1e-4).sum())
print(f"outputs with near-zero ensemble std on test (<1e-4): {near_zero_std} of {N_OUT} "
      f"({'members may have collapsed to the same solution -- check seeds/init' if near_zero_std else 'OK, members disagree meaningfully'})")
ensemble_spread

## Persist outputs

Saves, for Phase D: the ensemble predictions (row index in
`masters_data.xlsx`, observed values, ensemble mean and std, normalized
and physical), the member log, every member's epoch-by-epoch history,
the run configuration and the 10 trained models.

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

row_ids = np.arange(n)
pred_rows = []
for s in ["train", "val", "test"]:
    ids = row_ids[split == s]
    mean_s, std_s, _ = ens[s]
    for j in range(mean_s.shape[0]):
        row = {"row_id": int(ids[j]), "split": s}
        for i, out in enumerate(OUTPUT_COLS):
            row[f"{out}_obs"] = float(to_physical(truths[s][j, i], out))
            row[f"{out}_mean"] = float(to_physical(mean_s[j, i], out))
            row[f"{out}_std"] = float(to_physical_spread(std_s[j, i], out))
            row[f"{out}_mean_norm"] = float(mean_s[j, i])
            row[f"{out}_std_norm"] = float(std_s[j, i])
        pred_rows.append(row)
pl.DataFrame(pred_rows).write_csv(OUT_DIR / "C4_ensemble_predictions.csv")

hist_rows = []
for m, hist in enumerate(ensemble_histories):
    for k in range(len(hist["epoch"])):
        hist_rows.append({"member": m, **{key: hist[key][k] for key in hist}})
pl.DataFrame(hist_rows).write_csv(OUT_DIR / "C4_training_histories.csv")
metrics.write_csv(OUT_DIR / "C4_metrics.csv")

run_config = {
    "architecture_id": selection["architecture_id"], "hidden_units": list(HIDDEN_UNITS),
    "activation": hp["activation"], "hyperparameters_c3": hp, "physics_weights_used": scaled_w,
    "c2_calibration_mode": loss_config["calibration_mode"], "k_schedule": K_SCHEDULE, "t0_schedule": T0_SCHEDULE,
    "selection_start_epoch": SELECTION_START, "max_epochs": MAX_EPOCHS, "patience": PATIENCE,
    "lr_patience": LR_PATIENCE, "lr_factor": LR_FACTOR, "lbfgs_maxiter": LBFGS_MAXITER,
    "n_ensemble": N_ENSEMBLE, "ensemble_std": "sample (ddof=1)", "c1_config": C1_CONFIG,
}
with open(OUT_DIR / "C4_run_config.json", "w") as f:
    json.dump(run_config, f, indent=2, default=float)

for m, model in enumerate(ensemble_models):
    try:
        model.save(OUT_DIR / f"C4_ensemble_member_{m}.keras")
    except Exception as e:
        print(f"member {m}: '.keras' save failed ({e}), falling back to '.h5'")
        model.save(OUT_DIR / f"C4_ensemble_member_{m}.h5")
print(f"Saved to {OUT_DIR}")

## Next

- **B2 baseline** is trained with this notebook's protocol (same
  `train_member` machinery, every physics weight set to zero, 10 seeds,
  Adam + L-BFGS), so the Phase D comparison differs only in the physics
  terms.
- **Architecture sensitivity check:** train the PINN on one or two
  architectures B1 found statistically equivalent, to show the conclusion
  about the physics does not hinge on the backbone.
- **D1** loads `C4_ensemble_predictions.csv` alongside
  `B2_ensemble_predictions.csv` and runs the PINN-vs-baseline comparison
  (Eq. 3.21–3.27).